# SmartCHDR — Model 4: Vaccination Tracker

## Purpose
Rule-based vaccination tracker built on the Sri Lanka National
EPI (Expanded Programme on Immunization) schedule.

## How It Works
1. Takes child's date of birth as input
2. Calculates due date for every vaccine in Sri Lanka EPI schedule
3. Classifies each vaccine as: completed, due_soon, overdue, upcoming
4. Generates dashboard reminders for vaccines due within 14 days

## Input
- date_of_birth, today's date, completed vaccines list

## Output
- Full vaccination schedule with due dates and status
- Reminder alerts for upcoming vaccines

## Source
Sri Lanka National EPI Immunization Schedule
Ministry of Health, Sri Lanka

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
import json
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta

print("✅ Libraries imported!")

✅ Libraries imported!


In [2]:
import subprocess
subprocess.run(["pip", "install", "python-dateutil"], check=True)
print("✅ python-dateutil ready!")

✅ python-dateutil ready!


In [3]:
# Sri Lanka National Immunization Schedule
# Source: Ministry of Health, Nutrition and Indigenous Medicine
# Epidemiology Unit — Official 2017 Schedule (verified from official document)

EPI_SCHEDULE = [
    # ── FIRST YEAR OF LIFE ─────────────────────────────────────────────
    {
        'vaccine_id'  : 1,
        'vaccine_name': 'BCG',
        'due_months'  : 0,
        'due_label'   : '0-4 Weeks (At Birth)',
        'description' : 'Protects against Tuberculosis. Preferably '
                        'within 24 hours of birth before leaving hospital.',
        'is_critical' : True
    },
    {
        'vaccine_id'  : 2,
        'vaccine_name': 'OPV + Pentavalent (DTP-HepB-Hib) 1st Dose',
        'due_months'  : 2,
        'due_label'   : '2 Months',
        'description' : 'Oral Polio Vaccine + Pentavalent vaccine '
                        '(Diphtheria, Tetanus, Pertussis, Hepatitis B, '
                        'Haemophilus influenzae b) — 1st dose',
        'is_critical' : True
    },
    {
        'vaccine_id'  : 3,
        'vaccine_name': 'fIPV (Fractional IPV) 1st Dose',
        'due_months'  : 2,
        'due_label'   : '2 Months',
        'description' : 'Fractional Inactivated Polio Vaccine — 1st dose',
        'is_critical' : True
    },
    {
        'vaccine_id'  : 4,
        'vaccine_name': 'OPV + Pentavalent (DTP-HepB-Hib) 2nd Dose',
        'due_months'  : 4,
        'due_label'   : '4 Months',
        'description' : 'Oral Polio Vaccine + Pentavalent vaccine — 2nd dose',
        'is_critical' : True
    },
    {
        'vaccine_id'  : 5,
        'vaccine_name': 'fIPV (Fractional IPV) 2nd Dose',
        'due_months'  : 4,
        'due_label'   : '4 Months',
        'description' : 'Fractional Inactivated Polio Vaccine — 2nd dose',
        'is_critical' : True
    },
    {
        'vaccine_id'  : 6,
        'vaccine_name': 'OPV + Pentavalent (DTP-HepB-Hib) 3rd Dose',
        'due_months'  : 6,
        'due_label'   : '6 Months',
        'description' : 'Oral Polio Vaccine + Pentavalent vaccine — 3rd dose',
        'is_critical' : True
    },
    {
        'vaccine_id'  : 7,
        'vaccine_name': 'MMR 1st Dose',
        'due_months'  : 9,
        'due_label'   : '9 Months',
        'description' : 'Measles, Mumps and Rubella vaccine — 1st dose',
        'is_critical' : True
    },

    # ── SECOND YEAR OF LIFE ────────────────────────────────────────────
    {
        'vaccine_id'  : 8,
        'vaccine_name': 'Live JE (Japanese Encephalitis)',
        'due_months'  : 12,
        'due_label'   : '12 Months',
        'description' : 'Live Japanese Encephalitis vaccine',
        'is_critical' : True
    },
    {
        'vaccine_id'  : 9,
        'vaccine_name': 'OPV + DTP 4th Dose',
        'due_months'  : 18,
        'due_label'   : '18 Months',
        'description' : 'Oral Polio Vaccine + DTP booster — 4th dose',
        'is_critical' : True
    },

    # ── PRE-SCHOOL AGE ─────────────────────────────────────────────────
    {
        'vaccine_id'  : 10,
        'vaccine_name': 'MMR 2nd Dose',
        'due_months'  : 36,
        'due_label'   : '3 Years',
        'description' : 'Measles, Mumps and Rubella vaccine — 2nd dose',
        'is_critical' : True
    },

    # ── SCHOOL GOING AGE ───────────────────────────────────────────────
    {
        'vaccine_id'  : 11,
        'vaccine_name': 'OPV + DT 5th Dose',
        'due_months'  : 60,
        'due_label'   : '5 Years',
        'description' : 'Oral Polio Vaccine + Diphtheria/Tetanus — '
                        '5th dose booster',
        'is_critical' : True
    },
]

schedule_df = pd.DataFrame(EPI_SCHEDULE)
print("✅ CORRECTED Sri Lanka EPI Schedule:")
print(schedule_df[['vaccine_id','vaccine_name',
                    'due_label','description']].to_string(index=False))
print(f"\nTotal vaccines: {len(schedule_df)}")

✅ CORRECTED Sri Lanka EPI Schedule:
 vaccine_id                              vaccine_name            due_label                                                                                                                 description
          1                                       BCG 0-4 Weeks (At Birth)                                 Protects against Tuberculosis. Preferably within 24 hours of birth before leaving hospital.
          2 OPV + Pentavalent (DTP-HepB-Hib) 1st Dose             2 Months Oral Polio Vaccine + Pentavalent vaccine (Diphtheria, Tetanus, Pertussis, Hepatitis B, Haemophilus influenzae b) — 1st dose
          3            fIPV (Fractional IPV) 1st Dose             2 Months                                                                             Fractional Inactivated Polio Vaccine — 1st dose
          4 OPV + Pentavalent (DTP-HepB-Hib) 2nd Dose             4 Months                                                                         Oral Polio Vaccine + 

In [4]:
def calculate_vaccine_due_date(date_of_birth, due_months):
    """Calculate exact due date by adding months to date of birth."""
    return date_of_birth + relativedelta(months=due_months)

def classify_vaccine_status(due_date, today, is_completed):
    """
    Classify vaccine status based on due date and today's date.
    
    Status types:
    - completed  : parent has marked this vaccine as given
    - due_soon   : due within next 14 days (trigger reminder)
    - overdue    : due date has passed and not completed
    - upcoming   : due date is more than 14 days away
    """
    if is_completed:
        return 'completed'
    
    days_until_due = (due_date - today).days
    
    if days_until_due < 0:
        return 'overdue'
    elif days_until_due <= 14:
        return 'due_soon'
    else:
        return 'upcoming'

def get_vaccination_schedule(date_of_birth_str, 
                              completed_vaccine_ids=None,
                              today_str=None):
    """
    Generate full vaccination schedule for a child.
    
    Parameters:
    - date_of_birth_str : child's DOB as 'YYYY-MM-DD' string
    - completed_vaccine_ids : list of vaccine_ids already given
    - today_str : today's date as 'YYYY-MM-DD' (optional, 
                  defaults to actual today)
    
    Returns:
    - DataFrame with full schedule and statuses
    - List of reminder alerts
    """
    
    # Parse dates
    dob   = datetime.strptime(
        date_of_birth_str, '%Y-%m-%d').date()
    today = (datetime.strptime(today_str, '%Y-%m-%d').date() 
             if today_str 
             else date.today())
    
    if completed_vaccine_ids is None:
        completed_vaccine_ids = []
    
    # Calculate child's current age in months
    age_months = (today.year - dob.year) * 12 + \
                 (today.month - dob.month)
    
    schedule = []
    reminders = []
    
    for _, vaccine in schedule_df.iterrows():
        due_date    = calculate_vaccine_due_date(
            dob, vaccine['due_months'])
        is_completed = vaccine['vaccine_id'] in completed_vaccine_ids
        status      = classify_vaccine_status(
            due_date, today, is_completed)
        days_until  = (due_date - today).days
        
        entry = {
            'vaccine_id'  : vaccine['vaccine_id'],
            'vaccine_name': vaccine['vaccine_name'],
            'due_label'   : vaccine['due_label'],
            'due_date'    : due_date.strftime('%Y-%m-%d'),
            'status'      : status,
            'days_until'  : days_until,
            'description' : vaccine['description'],
            'is_critical' : vaccine['is_critical']
        }
        schedule.append(entry)
        
        # Generate reminder if due_soon or overdue
        if status == 'due_soon':
            reminders.append({
                'type'        : 'reminder',
                'vaccine_name': vaccine['vaccine_name'],
                'due_date'    : due_date.strftime('%Y-%m-%d'),
                'days_until'  : days_until,
                'message'     : f"⚠️ {vaccine['vaccine_name']} is "
                                f"due in {days_until} days "
                                f"({due_date.strftime('%d %b %Y')})"
            })
        elif status == 'overdue':
            reminders.append({
                'type'        : 'urgent',
                'vaccine_name': vaccine['vaccine_name'],
                'due_date'    : due_date.strftime('%Y-%m-%d'),
                'days_until'  : days_until,
                'message'     : f"🚨 {vaccine['vaccine_name']} is "
                                f"OVERDUE by "
                                f"{abs(days_until)} days! "
                                f"Please visit your clinic."
            })
    
    return pd.DataFrame(schedule), reminders, age_months

# Test with a child born 8 months ago
test_dob = (date.today() - relativedelta(months=8)
            ).strftime('%Y-%m-%d')
# Assume first 3 birth vaccines are completed
completed = [1, 2, 3]

schedule, reminders, age = get_vaccination_schedule(
    test_dob, completed)

print(f"Child age: {age} months")
print(f"\nFull Vaccination Schedule:")
print(schedule[['vaccine_name','due_label',
                'due_date','status',
                'days_until']].to_string(index=False))

print(f"\nActive Reminders ({len(reminders)}):")
for r in reminders:
    print(f"  {r['message']}")

Child age: 8 months

Full Vaccination Schedule:
                             vaccine_name            due_label   due_date    status  days_until
                                      BCG 0-4 Weeks (At Birth) 2025-10-15 completed        -243
OPV + Pentavalent (DTP-HepB-Hib) 1st Dose             2 Months 2025-12-15 completed        -182
           fIPV (Fractional IPV) 1st Dose             2 Months 2025-12-15 completed        -182
OPV + Pentavalent (DTP-HepB-Hib) 2nd Dose             4 Months 2026-02-15   overdue        -120
           fIPV (Fractional IPV) 2nd Dose             4 Months 2026-02-15   overdue        -120
OPV + Pentavalent (DTP-HepB-Hib) 3rd Dose             6 Months 2026-04-15   overdue         -61
                             MMR 1st Dose             9 Months 2026-07-15  upcoming          30
          Live JE (Japanese Encephalitis)            12 Months 2026-10-15  upcoming         122
                       OPV + DTP 4th Dose            18 Months 2027-04-15  upcoming     

In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import LabelEncoder

def build_vaccination_decision_tree():
    """
    Train a Decision Tree to classify vaccination urgency.
    
    Input features:
    - days_until_due: days until vaccine is due
    - child_age_months: child's current age
    - is_critical: whether vaccine is critical
    
    Output:
    - urgency: immediate, high, medium, low
    """
    
    # Generate training data from rules
    training_data = []
    
    for days in range(-180, 365):
        for age in range(0, 61):
            for critical in [0, 1]:
                # Rule-based labels
                if days < 0 and critical == 1:
                    urgency = 'immediate'
                elif days < 0 and critical == 0:
                    urgency = 'high'
                elif 0 <= days <= 7 and critical == 1:
                    urgency = 'immediate'
                elif 0 <= days <= 7:
                    urgency = 'high'
                elif 8 <= days <= 14:
                    urgency = 'medium'
                elif days > 14:
                    urgency = 'low'
                else:
                    urgency = 'low'
                    
                training_data.append({
                    'days_until_due'  : days,
                    'child_age_months': age,
                    'is_critical'     : critical,
                    'urgency'         : urgency
                })
    
    dt_df = pd.DataFrame(training_data)
    
    le_urgency = LabelEncoder()
    dt_df['urgency_encoded'] = le_urgency.fit_transform(
        dt_df['urgency'])
    
    X = dt_df[['days_until_due',
               'child_age_months',
               'is_critical']].values
    y = dt_df['urgency_encoded'].values
    
    # Train Decision Tree
    dt = DecisionTreeClassifier(
        max_depth=5, random_state=42)
    dt.fit(X, y)
    
    # Show tree structure
    tree_rules = export_text(
        dt,
        feature_names=['days_until_due',
                       'child_age_months',
                       'is_critical'])
    print("Decision Tree Structure (first levels):")
    print('\n'.join(tree_rules.split('\n')[:20]))
    print("...")
    
    return dt, le_urgency

dt_model, le_urgency = build_vaccination_decision_tree()

# Test the decision tree
test_cases = [
    [-5,  8,  1, 'Critical vaccine overdue'],
    [3,   6,  1, 'Critical vaccine due in 3 days'],
    [10,  12, 1, 'Critical vaccine due in 10 days'],
    [30,  24, 0, 'Non-critical vaccine due in 30 days'],
    [120, 36, 1, 'Vaccine due in 4 months'],
]

print("\nDecision Tree Predictions:")
print(f"{'Scenario':45s} {'Urgency':10s}")
print("-" * 57)
for days, age, critical, label in test_cases:
    pred = dt_model.predict([[days, age, critical]])[0]
    urgency = le_urgency.inverse_transform([pred])[0]
    print(f"{label:45s} {urgency.upper():10s}")

Decision Tree Structure (first levels):
|--- days_until_due <= 14.50
|   |--- is_critical <= 0.50
|   |   |--- days_until_due <= 7.50
|   |   |   |--- class: 0
|   |   |--- days_until_due >  7.50
|   |   |   |--- class: 3
|   |--- is_critical >  0.50
|   |   |--- days_until_due <= 7.50
|   |   |   |--- class: 1
|   |   |--- days_until_due >  7.50
|   |   |   |--- class: 3
|--- days_until_due >  14.50
|   |--- class: 2

...

Decision Tree Predictions:
Scenario                                      Urgency   
---------------------------------------------------------
Critical vaccine overdue                      IMMEDIATE 
Critical vaccine due in 3 days                IMMEDIATE 
Critical vaccine due in 10 days               MEDIUM    
Non-critical vaccine due in 30 days           LOW       
Vaccine due in 4 months                       LOW       


In [8]:
test_profiles = [
    {
        'label'    : 'Newborn (1 week old)',
        'dob'      : (date.today() - timedelta(days=7)
                      ).strftime('%Y-%m-%d'),
        # Only BCG (id=1) given at birth
        'completed': [1],
    },
    {
        'label'    : 'Child aged 8 months',
        'dob'      : (date.today() - relativedelta(months=8)
                      ).strftime('%Y-%m-%d'),
        # BCG + all 2month + all 4month + all 6month vaccines done
        # ids: 1=BCG, 2=OPV+Penta1, 3=fIPV1, 4=OPV+Penta2,
        #      5=fIPV2, 6=OPV+Penta3
        'completed': [1, 2, 3, 4, 5, 6],
    },
    {
        'label'    : 'Child aged 20 months',
        'dob'      : (date.today() - relativedelta(months=20)
                      ).strftime('%Y-%m-%d'),
        # All up to MMR1 + JE done, OPV+DTP 4th dose NOT yet done
        # ids: 1-8 completed (BCG through Live JE)
        'completed': [1, 2, 3, 4, 5, 6, 7, 8],
    },
]

for profile in test_profiles:
    print("=" * 60)
    print(f"TEST: {profile['label']}")
    print("=" * 60)
    
    schedule, reminders, age = get_vaccination_schedule(
        profile['dob'],
        profile['completed']
    )
    
    print(f"Age: {age} months")
    print(f"\nSchedule breakdown:")
    for _, row in schedule.iterrows():
        icon = {'completed':'✅',
                'upcoming' :'📅',
                'due_soon' :'⚠️',
                'overdue'  :'🚨'}.get(row['status'], '?')
        print(f"  {icon} {row['vaccine_name']:45s} "
              f"{row['due_label']:15s} {row['status']}")
    
    print(f"\nActive Alerts ({len(reminders)}):")
    if reminders:
        for r in reminders:
            print(f"  {r['message']}")
    else:
        print("  No immediate alerts")
    print()

TEST: Newborn (1 week old)
Age: 0 months

Schedule breakdown:
  ✅ BCG                                           0-4 Weeks (At Birth) completed
  📅 OPV + Pentavalent (DTP-HepB-Hib) 1st Dose     2 Months        upcoming
  📅 fIPV (Fractional IPV) 1st Dose                2 Months        upcoming
  📅 OPV + Pentavalent (DTP-HepB-Hib) 2nd Dose     4 Months        upcoming
  📅 fIPV (Fractional IPV) 2nd Dose                4 Months        upcoming
  📅 OPV + Pentavalent (DTP-HepB-Hib) 3rd Dose     6 Months        upcoming
  📅 MMR 1st Dose                                  9 Months        upcoming
  📅 Live JE (Japanese Encephalitis)               12 Months       upcoming
  📅 OPV + DTP 4th Dose                            18 Months       upcoming
  📅 MMR 2nd Dose                                  3 Years         upcoming
  📅 OPV + DT 5th Dose                             5 Years         upcoming

Active Alerts (0):
  No immediate alerts

TEST: Child aged 8 months
Age: 8 months

Schedule breakdown:
  ✅

In [7]:
os.makedirs('../ml_models', exist_ok=True)

# Save EPI schedule
schedule_df.to_csv(
    '../ml_models/model4_epi_schedule.csv', index=False)

# Save Decision Tree
with open('../ml_models/model4_decision_tree.pkl', 'wb') as f:
    pickle.dump(dt_model, f)

# Save label encoder
with open('../ml_models/model4_urgency_encoder.pkl', 'wb') as f:
    pickle.dump(le_urgency, f)

# Save full schedule as JSON for Flask
with open('../ml_models/model4_epi_schedule.json', 'w') as f:
    json.dump(EPI_SCHEDULE, f, indent=2)

print("✅ Model 4 files saved to ml_models/:")
print("  - model4_epi_schedule.csv")
print("  - model4_epi_schedule.json")
print("  - model4_decision_tree.pkl")
print("  - model4_urgency_encoder.pkl")

for fname in ['model4_epi_schedule.csv',
              'model4_decision_tree.pkl',
              'model4_urgency_encoder.pkl',
              'model4_epi_schedule.json']:
    path = f'../ml_models/{fname}'
    size = os.path.getsize(path)
    print(f"  {fname}: {size/1024:.1f} KB")

✅ Model 4 files saved to ml_models/:
  - model4_epi_schedule.csv
  - model4_epi_schedule.json
  - model4_decision_tree.pkl
  - model4_urgency_encoder.pkl
  model4_epi_schedule.csv: 1.2 KB
  model4_decision_tree.pkl: 1.9 KB
  model4_urgency_encoder.pkl: 0.3 KB
  model4_epi_schedule.json: 2.7 KB


## Model 4 Complete ✅

### How It Works
1. Parent enters child's date of birth in the app
2. System calculates due date for all 10 EPI vaccines
3. Decision Tree classifies urgency: immediate/high/medium/low
4. Dashboard shows full schedule with colour-coded statuses
5. Reminders triggered automatically 14 days before each vaccine

### Vaccine Status Types
| Status | Meaning |
|---|---|
| completed | Parent has marked vaccine as given |
| due_soon | Due within 14 days — reminder shown |
| overdue | Past due date — urgent alert shown |
| upcoming | Due date more than 14 days away |

### Files Saved
| File | Purpose |
|---|---|
| model4_epi_schedule.csv | Full EPI schedule data |
| model4_epi_schedule.json | JSON version for Flask API |
| model4_decision_tree.pkl | Trained Decision Tree |
| model4_urgency_encoder.pkl | Label encoder for urgency |

### How Flask Will Use This
1. Load EPI schedule JSON at startup
2. Accept child DOB and completed vaccines from frontend
3. Calculate all due dates
4. Run Decision Tree for urgency classification
5. Return schedule + reminders as JSON to React frontend